# 01 ^合肥统计:无模型表和数字。

制作每个**数据来源**(无模式)表格和来自清洁月的数字
组别，这里不取决于装配的型号，所以这个笔记本可以运行在之前，之后
或独立于建模(`02_modelling.ipynb`_`03_model_results.ipynb`).

** 已执行的笔记本是报告**:表格和数字是内在的，没有任何内容。
写入磁盘。在 `CONFIG` 中设定 `SAVE_PAPER_ASSETS = True` ,以额外写入
出版物资产 (`.tex`表，300 - dpi PNG + SVG + PDF 数字，组数) 至
`results/paper/` .

| 产出 | 函数 | 资产干(当保存时) |
| :--- | :--- | :--- |
| 按纬度范围分列的人口(每公吨一个表) | `paper_tables.demographics_table` | `rhr_demographics_thinned.tex` , `hrv_demographics_original.tex` |
| 人口分布(年龄、性别、纬度、国家、小组A-F) | `pf.generate_figure_1` | `figure_1` |
| 季节性观测地图2×2 地图每公吨 | `pf.build_country_gender_validation` + `pf.plot_seasonal_maps_2x2` | `{metric}_validation_country_gender.csv` , `{metric}_actual_all_seasonal_map_2x2` |
| 用户内敏感度(每个用户月) | `pf.make_within_user_sensitivity` | `within_user_sensitivity_combined` |
| 微薄验证( 3 位数字) | `pf.generate_figure_a1/a2/a3` | `figure_a1..a3` |
| 缩放大小 | 内嵌 | `cohort_summary.json` |

RHR 组的稀疏化 与模拟笔记本完全一样的常规
( `sl.apply_density_based_thinning` ),所以表格和数字都描述了合适的组别。
文件访问通过`file_io`(`./data`或`$WPM_DATA_DIR`;参见README.

人口和稀释验证数字适用每个年龄的最低限度计算阈值，以及
纬度波段； 地图数字需要 `data/README.md` 中描述的参考文件。
`data/extra_data/` .

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import gc
import json
import os
import sys
import time

from IPython.display import HTML, Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # 图书馆模块(公开发布)
import cohort_io as cio
import file_io as fio
import paper_tables
import publication_figures as pf
import seasonality_lib as sl

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
THIN_RHR = True  # 基于密度的稀释与 02_modelling. ipynb
USER_FRAC = 1.0  # 装入后需要保持的用户的分数( E) 1.0 =每个人；< 1 快速运行).
RANDOM_STATE = 42
RHR_COL, HRV_COL = 'avg_resting_heart_rate_bpm', 'avg_hrv'
RHR_RANGE, HRV_RANGE = (30.0, 150.0), (
    5.0,
    250.0,
)  # 可信窗口( 用户摘要)
# 还将出版物资产(.tex、png/svg/pdf、cohort_summary.json)写成
# 成果/文件/?否则，这个笔记本就是报告。
SAVE_PAPER_ASSETS = False

data_dir = fio.get_data_dir()
extra_dir = fio.get_extra_dir(data_dir)
paper_dir = fio.get_paper_dir(data_dir) if SAVE_PAPER_ASSETS else None
if paper_dir:
  fio.make_dirs(paper_dir)
pf.setup_nature_style()
print(
    f'data: {data_dir}\npublication assets:'
    f' {paper_dir or "not saved (SAVE_PAPER_ASSETS=False)"}'
)


def show_table(label, latex):
  """Renders a generated LaTeX table inline (preview + collapsible source)."""
  preview = paper_tables.table_preview(latex)
  display(Markdown(f'**`{label}`** — {preview.attrs.get("caption", "")}'))
  display(HTML(preview.to_html(index=False)))
  if preview.attrs.get('notes'):
    display(Markdown(f'<small>{preview.attrs["notes"]}</small>'))
  display(
      HTML(
          '<details><summary>LaTeX source</summary><pre style="font-size:80%">'
          + latex.replace('&', '&amp;').replace('<', '&lt;')
          + '</pre></details>'
      )
  )

In [ ]:
# ----------------------------------------------------------------------------
# 装入干净的组
# ----------------------------------------------------------------------------
COLS = [
    'user_id',
    'activity_period',
    'gender',
    'age',
    'lat_e7',
    'lng_e7',
    'country_2_alpha_cd',
    'dl_monthly',
    'avg_temp_mean',
]


def load_clean(metric, value_col):
  """Loads one cohort through cohort_io (headers / units mapped to the code's column names)."""
  t0 = time.time()
  path = cio.resolve_input_path(data_dir, metric)
  df = cio.load_cohort(data_dir, metric, columns=COLS + [value_col], path=path)
  if USER_FRAC < 1.0:
    keep = pd.Series(df['user_id'].unique()).sample(
        frac=USER_FRAC, random_state=RANDOM_STATE
    )
    df = df[df['user_id'].isin(keep)].reset_index(drop=True)
  print(
      f'{metric.upper()}: {len(df):,} user-months / {df["user_id"].nunique():,}'
      f' users ({time.time() - t0:.0f}s)'
  )
  return df


df_rhr_raw = load_clean('rhr', RHR_COL)
df_hrv = load_clean('hrv', HRV_COL)
# 组的日历年(在表格标题中使用)。
COHORT_YEAR = int(
    pd.to_datetime(df_rhr_raw['activity_period']).dt.year.mode()[0]
)

In [ ]:
# ----------------------------------------------------------------------------
# RHR 变薄的时代 - 22 动脉 - 与建模笔记本完全相同
# ----------------------------------------------------------------------------
if THIN_RHR:
  df_rhr, thin_stats = sl.apply_density_based_thinning(
      df_rhr_raw,
      target_col=RHR_COL,
      val_min=RHR_RANGE[0],
      val_max=RHR_RANGE[1],
      random_state=RANDOM_STATE,
  )
  rhr_summary = thin_stats['user_summary_after']
else:
  df_rhr = df_rhr_raw
  rhr_summary = sl.get_user_summary(
      df_rhr[df_rhr[RHR_COL].between(*RHR_RANGE)], RHR_COL
  )
hrv_summary = sl.get_user_summary(
    df_hrv[df_hrv[HRV_COL].between(*HRV_RANGE)], HRV_COL
)

# 用户级登记册(主要纬度带、平均值、年龄、性别)
# 由人口和稀疏验证数字分享:每组人建造一次。
t0 = time.time()
rhr_registry = pf.build_user_registry(df_rhr, val_col=RHR_COL)
hrv_registry = pf.build_user_registry(df_hrv, val_col=HRV_COL)
raw_registry = (
    pf.build_user_registry(df_rhr_raw, val_col=RHR_COL) if THIN_RHR else None
)
print(f'User registries built in {time.time() - t0:.0f}s')

## 按纬度划分的参加人数

In [ ]:
show_table(
    'tab:rhr_demographics_thinned',
    paper_tables.demographics_table(
        rhr_summary, 'rhr', year=COHORT_YEAR, out_dir=paper_dir
    ),
)
show_table(
    'tab:hrv_demographics_original',
    paper_tables.demographics_table(
        hrv_summary, 'hrv', year=COHORT_YEAR, out_dir=paper_dir
    ),
)

# 卷之七·诸方(下)\诸方门目(下) 股 ⁇ (去皮尖) 股 ⁇ (去皮尖) 股 ⁇ (去皮尖) 股 ⁇ (去皮尖，去皮尖，去皮尖) 股 ⁇ (去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，去皮尖，细，用皮尖，细，细细细，为末。
cohort_summary = {
    'rhr': {
        'users': int(df_rhr['user_id'].nunique()),
        'user_months': int(len(df_rhr)),
        'thinned': bool(THIN_RHR),
        'users_before_thinning': int(df_rhr_raw['user_id'].nunique()),
        'user_months_before_thinning': int(len(df_rhr_raw)),
    },
    'hrv': {
        'users': int(df_hrv['user_id'].nunique()),
        'user_months': int(len(df_hrv)),
    },
    'user_frac': USER_FRAC,
    'generated': time.strftime('%Y-%m-%d %H:%M:%S'),
}
display(Markdown('**Cohort sizes**'))
display(pd.DataFrame({m: cohort_summary[m] for m in ('rhr', 'hrv')}).fillna(''))
if paper_dir:
  with fio.open_file(os.path.join(paper_dir, 'cohort_summary.json'), 'w') as f:
    json.dump(cohort_summary, f, indent=2)

## 人口分布(A-F组)

In [ ]:
fig_1, png_1, svg_1 = pf.generate_figure_1(
    df_rhr=df_rhr,
    df_hrv=df_hrv,
    output_dir=paper_dir,
    file_prefix='figure_1',
    rhr_col=RHR_COL,
    hrv_col=HRV_COL,
    rhr_registry=rhr_registry,
    hrv_registry=hrv_registry,
)
plt.show()
if png_1:
  print(f'Saved {png_1}, {svg_1}')
plt.close('all')

## 观察的季节性地图

季节性观察2×2每公分量的和弦，它们只需要国家××性别×月表示。
(c)从记忆中的组群中构建的观测值(不涉及模型);
保存为`{metric}_validation_country_gender.csv`,在资产保存时的数字旁边。

In [ ]:
# 国家xx性别x月表示不需要参考表;地图本身需要
# extra_data/(自然地球地理特征和UNSD国家表，见data/README.md.
validation = {
    metric: pf.build_country_gender_validation(
        data_dir, metric, out_dir=paper_dir, with_predictions=False, df=df
    )
    for metric, df in (('rhr', df_rhr), ('hrv', df_hrv))
}
if fio.file_exists(os.path.join(extra_dir, pf.WORLD_GEOJSON_FILE)):
  for metric in ('rhr', 'hrv'):
    pf.plot_seasonal_maps_2x2(
        data_dir,
        extra_dir,
        paper_dir,
        metrics=(metric,),
        validation_frames=validation,
    )
  plt.show()
  plt.close('all')
else:
  print(
      f'{pf.WORLD_GEOJSON_FILE} not found in {extra_dir}: map figures skipped'
  )

## 用户内敏感度

RHR和HRV用户内月与月之间的变化与日间长度的变化相比
温度(没有涉及模型;函数重新装入所需的列本身)。

In [ ]:
pf.make_within_user_sensitivity(data_dir, paper_dir)
plt.show()
plt.close('all')

## 线程验证( 之前 v 之后)

In [ ]:
if THIN_RHR:
  for fn in (
      pf.generate_figure_a1,
      pf.generate_figure_a2,
      pf.generate_figure_a3,
  ):
    _, png, svg = fn(
        df_raw=df_rhr_raw,
        df_thinned=df_rhr,
        output_dir=paper_dir,
        rhr_col=RHR_COL,
        raw_registry=raw_registry,
        thinned_registry=rhr_registry,
    )
    plt.show()
    if png:
      print(f'Saved {png}, {svg}')
  plt.close('all')
else:
  print('THIN_RHR is False: Figures A.1-A.3 need the thinned cohort.')
del df_rhr_raw, raw_registry
gc.collect()

** 下一个：** 与[`02_modelling.ipynb`](02_modelling.ipynb)(每公吨一次)的模型相匹配，
然后将模型生成的表格和数字重新生成
[`03_model_results.ipynb`](03_model_results.ipynb) .